# EDA：经营概览与 RFM 分层对照

先运行 `python src/preprocess.py`、`python src/rfm.py`、`python src/cluster.py`。业务当日为 delivered 最晚下单日（2018-08-29）。

In [ ]:
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt

DB = Path("../data/processed/ecommerce.duckdb")
con = duckdb.connect(str(DB), read_only=True)
print(con.execute("SELECT as_of_date, as_of_month FROM meta_asof").fetchdf())

In [ ]:
kpi = con.execute("SELECT * FROM kpi_snapshot").fetchdf()
trend = con.execute("SELECT * FROM sales_trend_monthly").fetchdf()
print(kpi.T)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(trend["order_month"], trend["gmv"], marker="o")
ax.set_title("Monthly GMV (valid orders)")
ax.set_xlabel("Month")
ax.set_ylabel("GMV")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

In [ ]:
cats = con.execute(
    "SELECT category, gmv, gmv_share FROM category_sales ORDER BY category_rank LIMIT 10"
).fetchdf()
states = con.execute(
    "SELECT customer_state, gmv FROM sales_by_state ORDER BY gmv DESC LIMIT 10"
).fetchdf()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].barh(cats["category"][::-1], cats["gmv"][::-1])
axes[0].set_title("Top 10 categories by GMV")
axes[1].barh(states["customer_state"][::-1], states["gmv"][::-1])
axes[1].set_title("Top 10 states by GMV")
fig.tight_layout()

In [ ]:
seg = con.execute("SELECT * FROM rfm_segment_summary").fetchdf()
print(seg.to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(seg["segment"], seg["customers"])
axes[0].set_title("Customers by RFM segment")
axes[0].tick_params(axis="x", rotation=25)
axes[1].pie(seg["customers"], labels=seg["segment"], autopct="%1.1f%%")
axes[1].set_title("Segment share")
fig.tight_layout()

In [ ]:
heat = con.execute(
    "SELECT r_score, m_score, customers FROM rfm_score_heatmap"
).fetchdf()
grid = heat.pivot(index="m_score", columns="r_score", values="customers").sort_index(ascending=False)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(grid.to_numpy(), cmap="YlOrRd")
ax.set_xticks(range(grid.shape[1]), grid.columns)
ax.set_yticks(range(grid.shape[0]), grid.index)
ax.set_xlabel("R score (5 = most recent)")
ax.set_ylabel("M score (5 = highest spend)")
ax.set_title("RFM score heatmap (customer count)")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()

In [ ]:
compare = con.execute(
    """
    SELECT segment, cluster_label, COUNT(*) AS customers
    FROM rfm_clusters
    GROUP BY 1, 2
    ORDER BY 1, 2
    """
).fetchdf()
print(
    compare.pivot(index="segment", columns="cluster_label", values="customers")
    .fillna(0)
    .astype(int)
    .to_string()
)
con.close()